In [0]:
for key in [
    "fs.azure.account.auth.type.stmscaidev2026.dfs.core.windows.net",
    "fs.azure.account.oauth.provider.type.stmscaidev2026.dfs.core.windows.net",
    "fs.azure.account.oauth2.client.id.stmscaidev2026.dfs.core.windows.net",
    "fs.azure.account.oauth2.client.secret.stmscaidev2026.dfs.core.windows.net",
    "fs.azure.account.oauth2.client.endpoint.stmscaidev2026.dfs.core.windows.net"
]:
    try:
        spark.conf.unset(key)
    except Exception:
        pass

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import col, to_timestamp, lit

# ---------------------------------------------------
# BASE PATHS
# ---------------------------------------------------
BASE_PATH = "abfss://datalake@stmscaidev2026.dfs.core.windows.net"

BRONZE_PATH = f"{BASE_PATH}/bronze/year=2026/month=07/day=04/"
SILVER_PATH = f"{BASE_PATH}/silver/telemetry/"
GOLD_PATH   = f"{BASE_PATH}/gold/telemetry_kpis/"

# ---------------------------------------------------
# READ BRONZE DATA
# ---------------------------------------------------
bronze_df = spark.read.json(BRONZE_PATH)

# ---------------------------------------------------
# EXPECTED SILVER SCHEMA (REFERENCE ONLY)
# ---------------------------------------------------
expected_schema = StructType([
    StructField("eventId", StringType(), True),
    StructField("vehicleId", StringType(), True),
    StructField("driverId", StringType(), True),

    StructField("batteryVoltage", DoubleType(), True),
    StructField("engineTemperatureC", DoubleType(), True),
    StructField("fuelLevelPercent", DoubleType(), True),
    StructField("heading", DoubleType(), True),

    StructField("ignition", BooleanType(), True),

    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),

    StructField("odometerKm", DoubleType(), True),
    StructField("speedKmh", DoubleType(), True),

    StructField("timestamp", TimestampType(), True)
])

# ---------------------------------------------------
# SILVER TRANSFORMATION (FLATTEN + TYPE CAST)
# ---------------------------------------------------
silver_df = bronze_df.select(
    col("eventId"),
    col("vehicleId"),
    col("driverId"),

    col("batteryVoltage").cast("double"),
    col("engineTemperatureC").cast("double"),
    col("fuelLevelPercent").cast("double"),
    col("heading").cast("double"),
    col("ignition").cast("boolean"),

    col("odometerKm").cast("double"),
    col("speedKmh").cast("double"),

    col("location.latitude").alias("latitude"),
    col("location.longitude").alias("longitude"),

    to_timestamp(col("timestamp")).alias("timestamp")
)

# ---------------------------------------------------
# CLEAN DATA
# ---------------------------------------------------
silver_df = silver_df.dropna(
    subset=["eventId", "vehicleId", "timestamp"]
)

# Add partition columns (IMPORTANT for Lakehouse structure)
silver_df = silver_df \
    .withColumn("year", lit(2026)) \
    .withColumn("month", lit(7)) \
    .withColumn("day", lit(4))

display(silver_df)

# ---------------------------------------------------
# WRITE SILVER (DELTA - CORRECT USAGE)
# IMPORTANT: WRITE TO ROOT ONLY (NO partition in path)
# ---------------------------------------------------
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("year", "month", "day")
    .save(SILVER_PATH)
)

# ---------------------------------------------------
# GOLD LAYER (AGGREGATED KPI DATA)
# ---------------------------------------------------
from pyspark.sql.functions import avg, max, min

gold_df = silver_df.groupBy(
    "vehicleId",
    "year", "month", "day"
).agg(
    avg("speedKmh").alias("avg_speed"),
    max("speedKmh").alias("max_speed"),
    avg("engineTemperatureC").alias("avg_engine_temp"),
    avg("fuelLevelPercent").alias("avg_fuel_level"),
    max("odometerKm").alias("max_odometer")
)

# ---------------------------------------------------
# WRITE GOLD (DELTA)
# ---------------------------------------------------
(
    gold_df.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("year", "month", "day")
    .save(GOLD_PATH)
)

eventId,vehicleId,driverId,batteryVoltage,engineTemperatureC,fuelLevelPercent,heading,ignition,odometerKm,speedKmh,latitude,longitude,timestamp,year,month,day
ba61f8b1-2ca6-4577-a92d-299c43fb1b69,CAR-002,DRV-105,13.66,94.7,24.0,323.0,true,14310.7,13.0,43.546078,-79.647879,2026-07-04T02:49:32.676Z,2026,7,4
42f24787-2dd5-4d3a-9d47-2222d6780942,CAR-001,DRV-104,13.34,99.5,31.0,32.0,false,81383.7,53.0,43.497365,-79.775005,2026-07-04T02:49:35.705Z,2026,7,4
e8ca5c29-655e-4354-b96f-dc8dfdf36a4b,CAR-004,DRV-105,12.79,92.2,69.0,198.0,false,22700.5,81.0,43.542094,-79.638513,2026-07-04T02:49:38.736Z,2026,7,4
92bb7a15-7bc5-4951-84c8-557e8c122eee,CAR-002,DRV-103,12.42,89.7,75.0,117.0,true,32976.3,120.0,43.525886,-79.657099,2026-07-04T02:49:41.764Z,2026,7,4
877704ee-f8ba-4763-b576-e931dfb7463f,CAR-004,DRV-104,13.66,78.8,95.0,11.0,true,62864.5,41.0,43.471694,-79.657918,2026-07-04T02:49:44.801Z,2026,7,4
bd20d8d1-d395-4bfa-ac65-b4e4e8047404,CAR-005,DRV-104,12.82,96.2,81.0,252.0,true,77183.3,77.0,43.522866,-79.750765,2026-07-04T02:49:47.834Z,2026,7,4
e9a58bdf-6978-48a8-aa69-88b92d7c3416,CAR-003,DRV-103,14.27,86.9,17.0,255.0,true,17283.5,40.0,43.521264,-79.695666,2026-07-04T02:49:50.863Z,2026,7,4
897fc27e-4af2-4308-ad77-b67209e976f9,CAR-004,DRV-104,13.48,94.9,89.0,89.0,false,69772.6,45.0,43.494478,-79.713439,2026-07-04T02:49:53.893Z,2026,7,4
edd10e20-6926-4a0b-8655-6af02a7c3b6c,CAR-004,DRV-105,12.25,102.8,40.0,157.0,false,58175.6,99.0,43.507252,-79.652297,2026-07-04T02:49:56.934Z,2026,7,4
f534b2f1-7615-41ce-be2c-e7957238bea6,CAR-002,DRV-103,13.16,98.8,27.0,151.0,true,35957.5,31.0,43.500561,-79.641499,2026-07-04T02:49:59.963Z,2026,7,4


In [0]:
BASE_PATH = "abfss://datalake@stmscaidev2026.dfs.core.windows.net"

# list table versions
display(spark.sql("DESCRIBE HISTORY delta.`abfss://datalake@stmscaidev2026.dfs.core.windows.net/silver/telemetry/`"))

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-07-07T11:37:28.000Z,145403129064914,aakumara6@gmail.com,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [""year"",""month"",""day""])",null,List(1207486043227300),75418b21-790a-4564-a1d6-5f03a3770386,0707-113645-zfp0iko5-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 347815, numDeletionVectorsRemoved -> 0, numOutputRows -> 4800, numOutputBytes -> 347815)",null,Databricks-Runtime/18.2.x-photon-scala2.13
0,2026-07-06T03:02:26.000Z,145403129064914,aakumara6@gmail.com,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [""year"",""month"",""day""])",null,List(1207486043227300),608897c0-d46f-49a1-85e7-4d3bc9b3380e,0706-014350-txez2dt7-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 4800, numOutputBytes -> 347815)",null,Databricks-Runtime/18.2.x-photon-scala2.13


In [0]:
dbutils.fs.rm(
    "abfss://datalake@stmscaidev2026.dfs.core.windows.net/silver/telemetry/",
    True
)

True

In [0]:
bronze_df = spark.read.json(
  "abfss://datalake@stmscaidev2026.dfs.core.windows.net/bronze/year=2026/month=07/day=04/"
)

#display(bronze_df)

bronze_df.printSchema()
bronze_df.show(5, truncate=False)

root
 |-- batteryVoltage: double (nullable = true)
 |-- driverId: string (nullable = true)
 |-- engineTemperatureC: double (nullable = true)
 |-- eventId: string (nullable = true)
 |-- fuelLevelPercent: long (nullable = true)
 |-- heading: long (nullable = true)
 |-- ignition: boolean (nullable = true)
 |-- location: struct (nullable = true)
 |    |-- latitude: double (nullable = true)
 |    |-- longitude: double (nullable = true)
 |-- odometerKm: double (nullable = true)
 |-- speedKmh: double (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- vehicleId: string (nullable = true)

+--------------+--------+------------------+------------------------------------+----------------+-------+--------+-----------------------+----------+--------+--------------------------------+---------+
|batteryVoltage|driverId|engineTemperatureC|eventId                             |fuelLevelPercent|heading|ignition|location               |odometerKm|speedKmh|timestamp                       |vehic